In [20]:
def porto(v0,v1):
    """
    count portofolio by simple return
    (Capinski & Zastawniak, Mathematics for Finance)

    parameters:
    v0 = portofolio at the time 0 
    v1 = portofolio at the time 1 
    """
    assert v0 > 0, "V(0) must be positive - assumption 1.2"
    simple_return = (v1 - v0)/v0
    return simple_return
porto(1000, 2000)

1.0

In [21]:
import math
def logporto(v0,v1):
    """
    calculate portofolio by log return
    formula : ln(v1 / v0)
    parameters:
    v0 = portofolio at the time 0 v(0)
    v1 = portofolio at the time 1 v(1)
    """
    assert v0 > 0, "V(0) must be positive (Assumption 1.2)"
    assert v1 > 0, "V(1) must be positive (Assumption 1.2)"
    log_return = math.log(v1 / v0)
    return log_return
logporto(1000, 2000)

0.6931471805599453

In [22]:
dealer_a = {
    'EUR_USD' : {'buy' : 1.0202, 'sell' : 1.0284},
    'GBP_USD' : {'buy' : 1.5718, 'sell' : 1.5844}
}
dealer_b = {
    'EUR_GBP' : {'buy' : 0.6324, 'sell': 0.6401},
    'USD_GBP' : {'buy': 0.6299, 'sell': 0.6375}
}

In [23]:
def currency_arbitrage(dealer_a, dealer_b, initial=1.0):
    "USD -> GBP -> EUR -> USD"
    gbp = initial/dealer_a['GBP_USD']['sell']
    eur = gbp/dealer_b['EUR_GBP']['buy']
    usd = eur*dealer_a['EUR_USD']['buy']
    profit = usd-initial
    if usd > initial:
        print(f"Arbitrage found: USD -> GBP -> EUR -> USD")
        print(f"Initial : ${initial:.4f}")
        print(f"Final   : ${usd:.4f}")
        print(f"Profit  : ${profit:.4f} ({(profit/initial)*100:.4f}%)")
    else:
        print("No arbitrage on this path")
    return usd
currency_arbitrage(dealer_a, dealer_b, initial=1000)

Arbitrage found: USD -> GBP -> EUR -> USD
Initial : $1000.0000
Final   : $1018.1895
Profit  : $18.1895 (1.8190%)


1018.1895236940946

In [24]:
def arbitrage(su, a, sd):
    """
    check no-arbitrage condition based on proposition 1.1
     (Capinski & Zastawniak, Mathematics for Finance)

     Parameters:
     su = float - stock price in up scenario s(1)
     sd = float - stock price in down scenario s(1)
     a = float - bonds price at the future a(1)
    """
    if sd < a < su:
        print("no arbitrage")
    elif a<=sd:
        print("arbitrage - buy stock, short bonds")
        print(f"V(1) up = {su - a}", f"V(1) down = {sd -a}")
    elif a>=su:
        print("arbitrage - buy bonds, short stock")
        print(f"V(1) up = {-su + a}", f"V(1) down = {-sd + a}")
    hasil = sd,a,su
    return hasil
arbitrage(125, 110, 105)  
arbitrage(125, 130, 105)  
arbitrage(125, 100, 105)  

no arbitrage
arbitrage - buy bonds, short stock
V(1) up = 5 V(1) down = 25
arbitrage - buy stock, short bonds
V(1) up = 25 V(1) down = 5


(105, 100, 125)

In [25]:
rates = {
    'USD': {
        'GBP': 1 / dealer_a['GBP_USD']['sell'],  # 0.6311 — dealer A
        'EUR': 1 / dealer_a['EUR_USD']['sell'],   # 0.9724 — dealer A
    },
    'GBP': {
        'USD': dealer_a['GBP_USD']['buy'],         # 1.5718 — dealer A
        'EUR': 1/dealer_b['EUR_GBP']['sell']                                  # lo isi
    },
    'EUR': {
        'USD': dealer_a['EUR_USD']['buy'],          # 1.0202 — dealer A
        'GBP': dealer_b['EUR_GBP']['buy']                                   # lo isi
    }
}

In [26]:
def check_all_arbitrage(rates, initial=1.0):
    """
    Detect arbitrage opportunities across all 3-currency paths.
    Based on Exercise 1.3 - Capinski & Zastawniak, Chapter 1.
    
    Parameters:
    -----------
    rates : dict — exchange rates between currencies
    initial : float — starting amount (default 1.0)
    
    Returns:
    --------
    None — prints all paths with profit/loss
    """
    currencies = ['USD', 'GBP', 'EUR']

    for start in currencies:
        for mid in currencies:
            for end in currencies:

                if start == mid or mid == end or start == end:
                    continue
                if mid not in rates[start] or mid not in rates[end] or start not in rates[end]:
                    continue
                step1 = initial * rates[start][mid]
                step2 = step1 * rates[mid][end]
                step3 = step2 * rates[end][start]
                profit = step3-initial
                path = f'{start}->{mid}->{end}->{start}'
                if step3 > initial:
                    print(f'arbitrage: {path} | final : {step3:.4f} | profit : {profit:.4f}')
                else:
                    print(f'no-arbitrage: {path} | loss: {profit:.4f}')
check_all_arbitrage(rates)

arbitrage: USD->GBP->EUR->USD | final : 1.0059 | profit : 0.0059
no-arbitrage: USD->EUR->GBP->USD | loss: -0.0334
no-arbitrage: GBP->USD->EUR->GBP | loss: -0.0334
arbitrage: GBP->EUR->USD->GBP | final : 1.0059 | profit : 0.0059
arbitrage: EUR->USD->GBP->EUR | final : 1.0059 | profit : 0.0059
no-arbitrage: EUR->GBP->USD->EUR | loss: -0.0334


a = risk free asset
s = risk asset
v = portovolio
kv = return portofolio

In [27]:
def expected_return(return_up, return_down, prob_up):
    """
    Calculate expected return E(K)
    
    Parameters:
    -----------
    returns : list — return in every scenario
    probabilities : list — probability in every skenario
    
    Returns:
    --------
    float : expected return
    """
    prob_down = 1 - prob_up
    assert 0 < prob_up < 1, "probability must between 0 and 1"
    expected = (return_up * prob_up) + (return_down * prob_down)
    return expected

In [36]:
return_up = porto(10000, 11750)
return_down = porto(10000, 9250)
ek = expected_return(return_up, return_down, 0.8)
print(f"expected return : {ek:.2%}")

expected return : 12.50%


In [35]:
def std_deviation(return_up, return_down, prob_up):
    """
    Calculate standard deviation (risk) for binomial model.
    Ref: Capinski & Zastawniak, Section 1.4
    
    Parameters:
    -----------
    return_up   : float — return if stock goes up
    return_down : float — return if stock goes down
    prob_up     : float — probability of going up
    
    Returns:
    --------
    float : standard deviation
    """
    prob_down = 1 - prob_up
    ek = expected_return(return_up, return_down, prob_up)
    variance = (return_up - ek)**2 * prob_up + (return_down - ek)**2 * prob_down
    deviasi = math.sqrt(variance)
    return deviasi
return_up = porto(10000, 11750)
return_down = porto(10000, 9250)
std = std_deviation(return_up, return_down, 0.8)
print(f"standard deviasi: {std:.2%}")

standard deviasi: 10.00%


In [37]:
# Exercise 1.4 - Risk and Return
# Data
v0 = 10000
v1_up = 11750
v1_down = 9250
prob_up = 0.8

# Calculations
k_up = porto(v0, v1_up)
k_down = porto(v0, v1_down)
e_k = expected_return(k_up, k_down, prob_up)
sigma = std_deviation(k_up, k_down, prob_up)

# Results
print(f"Return if up       : {k_up:.2%}")
print(f"Return if down     : {k_down:.2%}")
print(f"Expected return    : {e_k:.2%}")
print(f"Standard deviation : {sigma:.2%}")
print(f"Return range       : {(e_k - sigma):.2%} to {(e_k + sigma):.2%}")

Return if up       : 17.50%
Return if down     : -7.50%
Expected return    : 12.50%
Standard deviation : 10.00%
Return range       : 2.50% to 22.50%
